# Predicción del uso problemático de internet en niños y adolescentes
**Proyecto Innovación IA · MIAA 2026-2 · Entrega 2**

Se evalúa el desempeño de modelos para estimar el índice de severidad del uso problemático de internet (`sii`, categorías 0 a 3) a partir de variables demográficas, físicas, de sueño y de uso declarado de internet. El análisis comprende la caracterización de la población, la comparación de bloques de predictores y la evaluación de variantes de los modelos en la cohorte principal de 5–17 años.

Fuente de datos disponible en [Child Mind Institute — Problematic Internet Use (Kaggle)](https://www.kaggle.com/competitions/child-mind-institute-problematic-internet-use).


## 0. Configuración del análisis

La configuración principal incluye la cohorte de 5–17 años y las comparaciones por separado del aporte de sueño y de uso declarado de internet. Los rangos de 5–19 y 5–22 años se contemplan como análisis de sensibilidad. En Colab se configuró scikit-learn 1.8.0.


In [ ]:
# Parámetros de ejecución
FUENTE_DATOS = "KAGGLE"  # "KAGGLE" o "LOCAL"
ARCHIVO_TRAIN_LOCAL = "/content/train.csv"
ARCHIVO_DICCIONARIO_LOCAL = "/content/data_dictionary.csv"
COHORTES_EVALUAR = ["5_17"]  # opcional: ["5_17", "5_19", "5_22"]
INCLUIR_ABLACIONES = True
DESCARGAR_ZIP_AL_FINAL = False  # Descarga opcional del ZIP en Colab
FORZAR_DESCARGA_DATOS = False

from pathlib import Path
import os, sys, platform, json, zipfile, subprocess, importlib.util, hashlib

EN_COLAB = importlib.util.find_spec("google.colab") is not None if importlib.util.find_spec("google") else False
RUTA_BASE = Path("/content" if EN_COLAB else Path.cwd())
RUTA_DATOS = RUTA_BASE / "datos_piu_miaa"
RUTA_RESULTADOS = RUTA_BASE / "resultados_piu_miaa"
RUTA_EDA = RUTA_RESULTADOS / "eda"
RUTA_MODELOS = RUTA_RESULTADOS / "modelos"
RUTA_ABLACION = RUTA_RESULTADOS / "ablaciones"
for ruta in (RUTA_DATOS,RUTA_RESULTADOS,RUTA_EDA,RUTA_MODELOS,RUTA_ABLACION):
    ruta.mkdir(parents=True,exist_ok=True)

if EN_COLAB:
    # Dependencias para la ejecución en Colab.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "kaggle>=1.7,<2", "scikit-learn==1.8.0", "statsmodels>=0.14,<0.16"],
                   check=True)

import numpy as np
import pandas as pd
import scipy
from scipy import stats
import sklearn
from statsmodels.stats.multitest import multipletests
import matplotlib.pyplot as plt
print("Entorno:",platform.python_version(),"| pandas",pd.__version__,"| scikit-learn",sklearn.__version__)
print("Directorio de resultados:",RUTA_RESULTADOS)

## 1. Conexión con Kaggle

La conexión con la API de Kaggle se autenticó mediante el token `KAGGLE_API_TOKEN`, configurado en Secretos de Google Colab.


In [ ]:
# Autenticación de Kaggle mediante Secretos de Colab.
if FUENTE_DATOS.upper() == "KAGGLE":
    if not EN_COLAB:
        raise RuntimeError(
            "La descarga autenticada requiere el secreto KAGGLE_API_TOKEN en Google Colab."
        )

    from google.colab import userdata

    try:
        token_temporal = userdata.get("KAGGLE_API_TOKEN")
    except Exception as exc:
        # Se informa el tipo de error sin incluir el detalle de la excepción.
        tipo_error = type(exc).__name__
        if tipo_error == "SecretNotFoundError":
            instruccion = (
                "El secreto KAGGLE_API_TOKEN no está registrado en Colab."
            )
        elif tipo_error == "NotebookAccessError":
            instruccion = (
                "El notebook no tiene acceso al secreto KAGGLE_API_TOKEN."
            )
        elif tipo_error in ("TimeoutException", "MessageError"):
            instruccion = (
                "El entorno de ejecución no pudo conectarse a Secretos de Colab."
            )
        else:
            instruccion = (
                "No se pudo consultar Secretos de Colab."
            )
        raise RuntimeError(
            f"No se pudo leer KAGGLE_API_TOKEN ({tipo_error}). {instruccion}"
        ) from None

    if not isinstance(token_temporal, str) or not token_temporal.strip():
        raise RuntimeError(
            "El secreto KAGGLE_API_TOKEN requiere un valor de texto no vacío."
        )

    os.environ["KAGGLE_API_TOKEN"] = token_temporal.strip()
    del token_temporal
    print("Credencial de Kaggle disponible.")

elif FUENTE_DATOS.upper() == "LOCAL":
    print("Fuente de datos: archivos locales.")
else:
    raise ValueError("FUENTE_DATOS debe ser KAGGLE o LOCAL.")


## 2. Carga de datos y definición de la cohorte

Se utilizaron `train.csv` y `data_dictionary.csv`. La carga admite la descarga desde Kaggle y el uso de copias locales. El conjunto original contiene 3.960 registros, 2.736 con `sii` disponible. De estos, 2.652 corresponden a la cohorte principal de 5–17 años.


In [ ]:
COMPETICION = "child-mind-institute-problematic-internet-use"

def descargar_archivo_competicion(nombre):
    destino = RUTA_DATOS / nombre
    if destino.exists() and not FORZAR_DESCARGA_DATOS:
        return destino
    # Se busca el ejecutable de Kaggle en PATH y en el directorio de scripts.
    import shutil
    import sysconfig
    ejecutable_kaggle = shutil.which("kaggle")
    if ejecutable_kaggle is None:
        candidato = Path(sysconfig.get_path("scripts")) / "kaggle"
        if candidato.is_file():
            ejecutable_kaggle = str(candidato)
    if ejecutable_kaggle is None:
        raise RuntimeError(
            "No se encontró el ejecutable de Kaggle. Se requiere instalar la dependencia."
        )
    cmd = [ejecutable_kaggle, "competitions", "download", COMPETICION,
           "-f", nombre, "-p", str(RUTA_DATOS)]
    if FORZAR_DESCARGA_DATOS:
        cmd.append("-o")  # fuerza la descarga cuando se solicita explícitamente
    ejecucion = subprocess.run(cmd, capture_output=True, text=True)
    if ejecucion.returncode != 0:
        detalle = (ejecucion.stderr or ejecucion.stdout).strip()[-700:]
        raise RuntimeError(
            f"Falló la descarga de {nombre} desde Kaggle. "
            "La descarga requiere permiso de acceso a los datos de la competición. "
            f"Respuesta del comando: {detalle or 'sin detalles'}"
        )
    if destino.exists():
        return destino
    # Algunas versiones de la API entregan el CSV como ZIP de un único archivo.
    candidatos = sorted(RUTA_DATOS.glob("*.zip"), key=lambda p:p.stat().st_mtime, reverse=True)
    for archivo_zip in candidatos:
        with zipfile.ZipFile(archivo_zip) as z:
            coincidencias = [x for x in z.namelist() if Path(x).name == nombre]
            if coincidencias:
                destino.write_bytes(z.read(coincidencias[0]))
                return destino
    raise FileNotFoundError(f"Kaggle no devolvió {nombre}; contenido en {RUTA_DATOS}: "
                            + ", ".join(p.name for p in RUTA_DATOS.iterdir()))

if FUENTE_DATOS.upper() == "KAGGLE":
    ruta_train = descargar_archivo_competicion("train.csv")
    ruta_diccionario = descargar_archivo_competicion("data_dictionary.csv")
elif FUENTE_DATOS.upper() == "LOCAL":
    ruta_train = Path(ARCHIVO_TRAIN_LOCAL).expanduser()
    ruta_diccionario = Path(ARCHIVO_DICCIONARIO_LOCAL).expanduser()
else:
    raise ValueError("FUENTE_DATOS debe ser KAGGLE o LOCAL")
for ruta in (ruta_train,ruta_diccionario):
    if not ruta.is_file():
        raise FileNotFoundError(f"No se encuentra {ruta}")
sha256_train = hashlib.sha256(ruta_train.read_bytes()).hexdigest()
print("Datos listos:",ruta_train.name,ruta_diccionario.name)
print("Huella SHA256 de train.csv:",sha256_train)

train = pd.read_csv(ruta_train)
diccionario = pd.read_csv(ruta_diccionario)
assert train.shape == (3960,82), f"Dimensiones inesperadas: {train.shape}"
assert train["id"].is_unique and train["sii"].notna().sum()==2736
campos_prohibidos = ["id","sii"]+[x for x in train.columns if x.startswith("PCIAT-")]
variables_eda = [x for x in train.columns if x not in campos_prohibidos]
assert len(variables_eda)==58, f"Se esperaban 58 variables no PCIAT; aparecen {len(variables_eda)}"
etiquetados = train.loc[train.sii.notna()].copy()
etiquetados["sii"] = etiquetados["sii"].astype(int)
etiquetados["grupo_edad"] = pd.cut(etiquetados["Basic_Demos-Age"],[0,12,17,200],labels=["5-12","13-17","18-22"])
principal = etiquetados.loc[etiquetados["Basic_Demos-Age"].between(5,17)].copy()
assert len(principal)==2652
assert principal.sii.value_counts().reindex([0,1,2,3],fill_value=0).tolist()==[1562,700,360,30]
print("Fuente original:",train.shape,"| etiquetados:",len(etiquetados),
      "| cohorte 5–17:",len(principal),"| variables auditadas:",len(variables_eda))
print("Distribución 5–17:",principal.sii.value_counts().sort_index().to_dict())
if sha256_train!="f3d32a58301631ba3897668fc167bcc6233b005e41c35c46a75586976d0737d7":
    print("La huella SHA-256 difiere de la copia utilizada en el informe.")

## 3. Distribución de la severidad y características de la población

El análisis exploratorio describe los 2.736 participantes de 5–22 años con `sii` disponible. Se presentan conteos por edad y categoría de severidad, junto con medianas de las mediciones observadas.


In [ ]:
SALIDA = RUTA_EDA
campos_clave = ['Basic_Demos-Age','Physical-BMI','SDS-SDS_Total_T','CGAS-CGAS_Score',
  'PAQ_A-PAQ_A_Total','PAQ_C-PAQ_C_Total','PreInt_EduHx-computerinternet_hoursday',
  'Physical-HeartRate','BIA-BIA_Fat']
filas = []
for sii, grupo in etiquetados.groupby('sii'):
    r = {'sii':int(sii),'n':len(grupo),
         'porcentaje_sexo_codigo_0':round(100*(grupo['Basic_Demos-Sex']==0).mean(),2)}
    for campo in campos_clave:
        r[campo+'__n']=int(grupo[campo].notna().sum())
        r[campo+'__mediana']=float(grupo[campo].median()) if grupo[campo].notna().any() else None
    filas.append(r)
medianas=pd.DataFrame(filas)
medianas.to_csv(SALIDA/'01_medidas_por_sii.csv',index=False)

edades = pd.crosstab(etiquetados.grupo_edad,etiquetados.sii).reindex(columns=[0,1,2,3])
edades['total']=edades.sum(axis=1)
edades.to_csv(SALIDA/'02_edad_por_sii.csv')

for c,nombre in [('Basic_Demos-Sex','sexo'),('PreInt_EduHx-computerinternet_hoursday','internet')]:
    tab = pd.crosstab(etiquetados.sii,etiquetados[c],dropna=False).reindex([0,1,2,3])
    tab.to_csv(SALIDA/f'03_{nombre}_conteos.csv')
    tab.div(tab.sum(axis=1),axis=0).mul(100).round(2).to_csv(SALIDA/f'04_{nombre}_porcentajes_incluyendo_faltantes.csv')
    tab_val=pd.crosstab(etiquetados.sii,etiquetados[c]).reindex([0,1,2,3])
    tab_val.div(tab_val.sum(axis=1),axis=0).mul(100).round(2).to_csv(SALIDA/f'04_{nombre}_porcentajes_respuestas_validas.csv')

print(medianas[['sii','n','Basic_Demos-Age__mediana','SDS-SDS_Total_T__mediana','Physical-BMI__mediana']].to_string(index=False))
print('\nEdad x sii (casos):\n',edades)

## 4. Datos faltantes y cobertura de los instrumentos

Se cuantificaron los datos faltantes por variable y nivel de severidad. La cobertura de los cuestionarios de actividad física PAQ-A y PAQ-C se interpreta según sus rangos de edad de aplicación.


In [ ]:
faltantes=[]
for c in variables_eda:
    r={'variable':c,'faltantes_global_pct':round(100*etiquetados[c].isna().mean(),2),
       'n_con_valor':int(etiquetados[c].notna().sum())}
    for sii,g in etiquetados.groupby('sii'):
        r[f'sii{sii}_faltantes_pct']=round(100*g[c].isna().mean(),2)
        r[f'sii{sii}_n_con_valor']=int(g[c].notna().sum())
    faltantes.append(r)
faltantes=pd.DataFrame(faltantes).sort_values('faltantes_global_pct',ascending=False)
faltantes.to_csv(SALIDA/'05_faltantes_por_sii.csv',index=False)

instrumentos={'Físico':[c for c in variables_eda if c.startswith('Physical-') and c!='Physical-Season'],
    'Bioimpedancia':[c for c in variables_eda if c.startswith('BIA-') and c!='BIA-Season'],
    'Sueño SDS':['SDS-SDS_Total_T','SDS-SDS_Total_Raw'],
    'PAQ-A':['PAQ_A-PAQ_A_Total'],'PAQ-C':['PAQ_C-PAQ_C_Total'],
    'Uso declarado internet':['PreInt_EduHx-computerinternet_hoursday'],
    'CGAS':['CGAS-CGAS_Score']}
coberturas=[]
for nombre,campos in instrumentos.items():
    for sii,g in etiquetados.groupby('sii'):
        coberturas.append({'instrumento':nombre,'sii':int(sii),'n':len(g),
            'con_al_menos_una_medida':int(g[campos].notna().any(axis=1).sum()),
            'faltantes_celdas_pct':round(100*g[campos].isna().to_numpy().mean(),2)})
pd.DataFrame(coberturas).to_csv(SALIDA/'06_cobertura_instrumentos.csv',index=False)
print('Promedio de celdas vacías en 58 campos no PCIAT:',round(100*etiquetados[variables_eda].isna().to_numpy().mean(),2),'%')
print(faltantes[['variable','faltantes_global_pct']].head(12).to_string(index=False))

## 5. Asociaciones exploratorias por edad

Se estimaron correlaciones de Spearman con los pares de observaciones disponibles. Para las correlaciones globales se ajustaron los valores p por comparaciones múltiples mediante Benjamini–Hochberg. El análisis por estratos etarios exploró cuánto variaban las asociaciones con la edad.


In [ ]:
asociaciones=[]
for c in variables_eda:
    if pd.api.types.is_numeric_dtype(etiquetados[c]) and etiquetados[c].notna().sum()>80 and etiquetados[c].nunique(dropna=True)>6:
        v=etiquetados[['sii',c]].dropna()
        rho,p=stats.spearmanr(v.sii,v[c])
        asociaciones.append({'variable':c,'n':len(v),'rho_spearman':float(rho),'p_exploratoria':float(p),
                             'n_sii3':int((v.sii==3).sum())})
# Se incluyen sexo y uso declarado de internet por su interés en el análisis.
for c in ['PreInt_EduHx-computerinternet_hoursday','Basic_Demos-Sex']:
    v=etiquetados[['sii',c]].dropna()
    rho,p=stats.spearmanr(v.sii,v[c])
    asociaciones.append({'variable':c,'n':len(v),'rho_spearman':float(rho),'p_exploratoria':float(p),
                         'n_sii3':int((v.sii==3).sum())})
asociaciones=pd.DataFrame(asociaciones)
asociaciones['q_fdr_bh']=multipletests(asociaciones.p_exploratoria,method='fdr_bh')[1]
asociaciones.sort_values('q_fdr_bh').to_csv(SALIDA/'07_asociaciones_exploratorias.csv',index=False)

estratos=[]
for et,grupo in etiquetados.groupby('grupo_edad',observed=True):
    for c in ['PreInt_EduHx-computerinternet_hoursday','SDS-SDS_Total_T','Physical-BMI',
              'CGAS-CGAS_Score','PAQ_A-PAQ_A_Total','PAQ_C-PAQ_C_Total']:
        v=grupo[['sii',c]].dropna()
        if len(v)>=30 and v.sii.nunique()>1 and v[c].nunique()>1:
            rho,p=stats.spearmanr(v.sii,v[c])
        else:rho,p=np.nan,np.nan
        estratos.append({'edad':str(et),'variable':c,'n':len(v),'n_sii3':int((v.sii==3).sum()),
                         'rho_spearman':rho,'p_exploratoria':p})
pd.DataFrame(estratos).to_csv(SALIDA/'08_asociaciones_por_edad.csv',index=False)
print(asociaciones[asociaciones.variable.isin(['Basic_Demos-Age','SDS-SDS_Total_T','Physical-BMI',
 'PreInt_EduHx-computerinternet_hoursday','PAQ_A-PAQ_A_Total','PAQ_C-PAQ_C_Total'])][
 ['variable','n','rho_spearman','q_fdr_bh']].round(4).to_string(index=False))

## 6. Distribuciones de edad, sueño y uso declarado de internet

Se compararon las distribuciones por nivel de severidad. La escala de uso declarado de internet se expresa en cuatro categorías ordinales (0–3) y sus proporciones se calcularon entre quienes respondieron.


In [ ]:
plt.figure(figsize=(8,4.5))
for sii in [0,1,2,3]:
    edades_s=etiquetados.loc[etiquetados.sii==sii,'Basic_Demos-Age']
    plt.hist(edades_s,bins=np.arange(5,24),alpha=.5,density=True,label=f'sii={sii} (n={len(edades_s)})')
plt.title('Edad por categoría de severidad');plt.xlabel('Edad, años');plt.ylabel('Densidad relativa');plt.legend();plt.tight_layout()
plt.savefig(SALIDA/'grafico_edad_sii.png',dpi=150);plt.show();plt.close()

r=pd.crosstab(etiquetados.sii,etiquetados['PreInt_EduHx-computerinternet_hoursday'],normalize='index').reindex([0,1,2,3]).fillna(0)
r.plot(kind='bar',stacked=True,figsize=(8,4),colormap='viridis')
plt.title('Uso declarado de internet por sii (respuestas disponibles)');plt.xlabel('sii');plt.ylabel('Proporción');plt.legend(title='Código de uso');plt.tight_layout()
plt.savefig(SALIDA/'grafico_internet_sii.png',dpi=150);plt.show();plt.close()

vals=[etiquetados.loc[etiquetados.sii==i,'SDS-SDS_Total_T'].dropna() for i in [0,1,2,3]]
plt.figure(figsize=(6,4));plt.boxplot(vals,tick_labels=['0','1','2','3'],showfliers=False)
plt.ylabel('SDS T');plt.xlabel('sii');plt.title('Puntuación de sueño SDS por sii');plt.tight_layout()
plt.savefig(SALIDA/'grafico_sds_sii.png',dpi=150);plt.show();plt.close()

## 7. Modelos de referencia y bloques de predictores

| Bloque | Variables |
|---|---|
| A · Demográfico | Edad y sexo |
| B · Físico | Bloque A, índice de masa corporal, frecuencia cardiaca y presión arterial sistólica y diastólica |
| C · Ampliado | Bloque B, puntuación T de sueño (SDS) y categoría de uso declarado de internet |

Se compararon la regresión logística multiclase y el *boosting* por histogramas con hiperparámetros fijos, usando las mismas particiones de validación cruzada estratificada de cinco pliegues y tres repeticiones. La imputación, la codificación y el escalado se ajustaron dentro de cada partición de entrenamiento. Los valores no positivos de índice de masa corporal, frecuencia cardiaca y presión arterial se trataron como faltantes. Se excluyeron de los predictores el identificador, el desenlace y los ítems PCIAT.


In [ ]:
# Configuración de los modelos de referencia.
os.environ["MIAA_BASE"] = str(RUTA_DATOS)
os.environ["MIAA_SOURCE_CSV"] = str(ruta_train)
os.environ["MIAA_OUTPUT"] = str(RUTA_MODELOS)
os.environ["MIAA_COHORTS"] = ",".join(COHORTES_EVALUAR)

from pathlib import Path
import json, sys, os, platform
import numpy as np
import pandas as pd
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import cohen_kappa_score, f1_score, balanced_accuracy_score, confusion_matrix, recall_score

BASE = Path(os.environ.get('MIAA_BASE', '/mnt/data'))
SOURCE = Path(os.environ.get('MIAA_SOURCE_CSV', BASE / 'train.csv'))
OUTPUT = Path(os.environ.get('MIAA_OUTPUT', BASE / 'Modelos_referencia_PIU_resultados'))
OUTPUT.mkdir(parents=True,exist_ok=True)
SEED=2026
NUM_SPLITS=5
NUM_REPEATS=3
BLOCKS={
    'A_demografia':['Basic_Demos-Age','Basic_Demos-Sex'],
    'B_fisica':['Basic_Demos-Age','Basic_Demos-Sex','Physical-BMI','Physical-HeartRate','Physical-Systolic_BP','Physical-Diastolic_BP'],
    'C_ampliada':['Basic_Demos-Age','Basic_Demos-Sex','Physical-BMI','Physical-HeartRate','Physical-Systolic_BP','Physical-Diastolic_BP','SDS-SDS_Total_T','PreInt_EduHx-computerinternet_hoursday'],
}
COHORTS={'5_17':(5,17),'5_19':(5,19),'5_22':(5,22)}
CATEGORICAL={'Basic_Demos-Sex','PreInt_EduHx-computerinternet_hoursday'}

# El control de valores no positivos se aplica de forma independiente del desenlace.
def basic_quality_clean(df):
    df=df.copy()
    for col in ('Physical-BMI','Physical-HeartRate','Physical-Systolic_BP','Physical-Diastolic_BP'):
        if col in df: df.loc[df[col] <= 0, col]=np.nan
    return df

def mk_pipeline(features, model_name):
    nums=[x for x in features if x not in CATEGORICAL]
    cats=[x for x in features if x in CATEGORICAL]
    numeric_pipe=Pipeline([('imputer',SimpleImputer(strategy='median',add_indicator=True,keep_empty_features=True)),('scaler',StandardScaler())])
    categorical_pipe=Pipeline([('imputer',SimpleImputer(strategy='most_frequent',keep_empty_features=True)),('one_hot',OneHotEncoder(handle_unknown='ignore',sparse_output=False))])
    transform=ColumnTransformer([('numeric',numeric_pipe,nums),('category',categorical_pipe,cats)],remainder='drop',sparse_threshold=0)
    if model_name=='Logistica_multiclase':
        model=LogisticRegression(C=1.0, class_weight='balanced',max_iter=1800,random_state=SEED)
    elif model_name=='Boosting_histogramas':
        model=HistGradientBoostingClassifier(max_iter=110,learning_rate=.07,max_leaf_nodes=10,min_samples_leaf=20,l2_regularization=1.0,random_state=SEED,class_weight='balanced',early_stopping=False)
    else: raise ValueError(model_name)
    return Pipeline([('preprocess',transform),('model',model)])

def main():
    df=pd.read_csv(SOURCE)
    assert df.shape[0]==3960, f'Esperados 3960, recibidos {len(df)}'
    assert df['id'].is_unique
    assert df['sii'].notna().sum()==2736
    assert not [c for v in BLOCKS.values() for c in v if c.startswith('PCIAT-') or c in ('sii','id')]
    for cols in BLOCKS.values(): assert set(cols).issubset(df.columns)
    df=basic_quality_clean(df)
    rows=[]; confs=[]; cohort_comp=[]; conf_aggregate=[]
    requested=os.environ.get('MIAA_COHORTS', '5_17').split(',')
    for cohort,(lo,hi) in COHORTS.items():
        if cohort not in requested: continue
        data=df.loc[df['sii'].notna() & df['Basic_Demos-Age'].between(lo,hi)].copy()
        y=data['sii'].astype(int).to_numpy()
        cohort_comp.append({'cohorte':cohort,'n':len(y),**{f'sii_{v}':int(sum(y==v)) for v in range(4)}})
        assert min(np.bincount(y, minlength=4))>=NUM_SPLITS
        splits=list(RepeatedStratifiedKFold(n_splits=NUM_SPLITS,n_repeats=NUM_REPEATS,random_state=SEED).split(data,y))
        for block,features in BLOCKS.items():
            x=data[features]
            for model_name in ('Logistica_multiclase','Boosting_histogramas'):
                for k,(train_idx,valid_idx) in enumerate(splits):
                    pipe=mk_pipeline(features,model_name)
                    pipe.fit(x.iloc[train_idx],y[train_idx])
                    predicted=pipe.predict(x.iloc[valid_idx])
                    truth=y[valid_idx]
                    cm=confusion_matrix(truth,predicted,labels=[0,1,2,3]); conf_aggregate.append({'cohorte':cohort,'bloque':block,'modelo':model_name,'repeticion':k//NUM_SPLITS+1,'fold':k%NUM_SPLITS+1,'cm':cm})
                    entry={'cohorte':cohort,'bloque':block,'modelo':model_name,'repeticion':k//NUM_SPLITS+1,'fold':k%NUM_SPLITS+1,'n_validacion':len(truth),'n_train':len(train_idx),
                           'qwk':float(cohen_kappa_score(truth,predicted,weights='quadratic',labels=[0,1,2,3])),
                           'f1_macro':float(f1_score(truth,predicted,labels=[0,1,2,3],average='macro',zero_division=0)),
                           'balanced_accuracy':float(balanced_accuracy_score(truth,predicted))}
                    for v in range(4):
                        entry[f'n_sii_{v}']=int(sum(truth==v))
                        entry[f'pred_sii_{v}']=int(sum(predicted==v))
                        entry[f'recall_sii_{v}']=float(recall_score(truth,predicted,labels=[0,1,2,3],average=None,zero_division=0)[v])
                    rows.append(entry)
    folds=pd.DataFrame(rows)
    group_cols=['cohorte','bloque','modelo']
    summary=folds.groupby(group_cols,sort=False).agg(
        n_folds=('qwk','size'),qwk_media=('qwk','mean'),qwk_sd=('qwk','std'),qwk_mediana=('qwk','median'),qwk_min=('qwk','min'),qwk_max=('qwk','max'),f1_macro_media=('f1_macro','mean'),f1_macro_sd=('f1_macro','std'),acc_balanceada_media=('balanced_accuracy','mean'),
        **{f'recall_sii_{i}_medio':(f'recall_sii_{i}','mean') for i in range(4)}
    ).reset_index()
    for gkey,group in folds.groupby(group_cols,sort=False):
        for v in range(4):
            pass
        key=dict(zip(group_cols,gkey))
        matching=[c['cm'] for c in conf_aggregate if all(c[k]==q for k,q in key.items())]
        cm=sum(matching)
        for i in range(4):
            for j in range(4): confs.append({**key,'real':i,'predicho':j,'conteo_en_3_repeticiones':int(cm[i,j])})
    folds.to_csv(OUTPUT/'01_metricas_por_fold.csv',index=False)
    summary.to_csv(OUTPUT/'02_resumen_modelos.csv',index=False)
    pd.DataFrame(confs).to_csv(OUTPUT/'03_matrices_confusion_3_repeticiones.csv',index=False)
    pd.DataFrame(cohort_comp).to_csv(OUTPUT/'04_cohortes.csv',index=False)
    assert folds.shape[0]==len(requested)*len(BLOCKS)*2*NUM_SPLITS*NUM_REPEATS
    assert folds['n_sii_3'].min()>=5
    meta={
    'fuente':'Child Mind Institute - Problematic Internet Use, train.csv', 'n_total_original':3960, 'n_con_sii':2736,
    'bloques_variables':BLOCKS,'cohortes':COHORTS,'algoritmos':['Logistica_multiclase','Boosting_histogramas'],
    'cv':{'tipo':'RepeatedStratifiedKFold','n_splits':NUM_SPLITS,'n_repeats':NUM_REPEATS,'random_state':SEED},
    'saneamiento_deterministico':'Solo valores <=0 de BMI, FC y PA se reemplazan por NaN; resto de extremos se mantienen; sin mirar sii.',
    'preprocesamiento':'Imputacion mediana + indicadores de ausencias + escalado numerico; imputacion moda + onehot categorico; fit exclusivamente en train folds',
    'modelos_hiperparametros':{'Logistica_multiclase':{'C':1,'max_iter':1800,'class_weight':'balanced'},'Boosting_histogramas':{'max_iter':110,'learning_rate':.07,'max_leaf_nodes':10,'min_samples_leaf':20,'l2_regularization':1.,'class_weight':'balanced','early_stopping':False}},
    'evaluacion':'QWK/F1-macro/media por fold; matrices acumuladas sobre 3 repeticiones, no participantes independientes.',
    'precauciones':['Sin test independiente','Hiperparametros fijados antes, no optimizados','No extrapolar a prediccion prospectiva','No se presentan intervalos de confianza independientes a partir de folds repetidos','No usar test.csv publico solapado con train.csv'],
    'python':platform.python_version(), 'pandas':pd.__version__, 'numpy':np.__version__}
    import sklearn;meta['scikit_learn']=sklearn.__version__
    (OUTPUT/'05_metodologia_y_versiones.json').write_text(json.dumps(meta,indent=2,ensure_ascii=False),encoding='utf-8')
    print('RESULTADOS\n',pd.DataFrame(cohort_comp).to_string(index=False));print('\n',summary[['cohorte','bloque','modelo','qwk_media','qwk_sd','f1_macro_media','recall_sii_3_medio']].round(3).to_string(index=False))
    return summary

resumen_referencia = main()

## 8. Aporte de las variables de sueño e internet

Tras revisar el bloque C, se evaluaron por separado el sueño y el uso declarado de internet al añadir cada variable al bloque B. En la cohorte de 5–17 años se compararon las diferencias de QWK en los mismos pliegues y con los mismos modelos. El análisis tuvo carácter exploratorio.


In [ ]:
from sklearn.metrics import precision_score
from sklearn.dummy import DummyClassifier

if INCLUIR_ABLACIONES:
    df_ab = basic_quality_clean(train)
    data_ab = df_ab.loc[df_ab["sii"].notna() & df_ab["Basic_Demos-Age"].between(5,17)].copy()
    y_ab = data_ab["sii"].astype(int).to_numpy()
    assert len(y_ab)==2652 and np.bincount(y_ab,minlength=4).tolist()==[1562,700,360,30]
    splits_ab = list(RepeatedStratifiedKFold(n_splits=5,n_repeats=3,random_state=2026).split(data_ab,y_ab))
    bloques_ab = {
        "B_mas_SDS": BLOCKS["B_fisica"]+["SDS-SDS_Total_T"],
        "B_mas_Internet": BLOCKS["B_fisica"]+["PreInt_EduHx-computerinternet_hoursday"]
    }
    registros_ab=[]
    matrices_ab={}
    for bloque,columnas in bloques_ab.items():
        for modelo in ("Logistica_multiclase","Boosting_histogramas"):
            cm_total = np.zeros((4,4),dtype=int)
            for i,(ind_ent,ind_val) in enumerate(splits_ab):
                clasificador = mk_pipeline(columnas,modelo)
                clasificador.fit(data_ab[columnas].iloc[ind_ent],y_ab[ind_ent])
                pred = clasificador.predict(data_ab[columnas].iloc[ind_val])
                real = y_ab[ind_val]
                cm_total += confusion_matrix(real,pred,labels=[0,1,2,3])
                registros_ab.append({
                    "cohorte":"5_17", "bloque":bloque,"modelo":modelo,
                    "repeticion": i//5 +1, "fold":i%5+1,
                    "qwk":float(cohen_kappa_score(real,pred,weights="quadratic",labels=[0,1,2,3])),
                    "f1_macro":float(f1_score(real,pred,average="macro",labels=[0,1,2,3],zero_division=0)),
                    "balanced_accuracy":float(balanced_accuracy_score(real,pred)),
                    "recall_sii_3":float(recall_score(real,pred,labels=[0,1,2,3],average=None,zero_division=0)[3])
                })
            matrices_ab[(bloque,modelo)] = cm_total
            print("Ablación finalizada:",bloque,"/",modelo)
    ab_folds = pd.DataFrame(registros_ab)
    ab_folds.to_csv(RUTA_ABLACION/"01_metricas_ablation_folds.csv",index=False)

    base_folds = pd.read_csv(RUTA_MODELOS/"01_metricas_por_fold.csv").query('cohorte=="5_17"')
    comparar = pd.concat([base_folds,ab_folds],ignore_index=True,sort=False)
    resumen_total = comparar.groupby(["bloque","modelo"],sort=False).agg(
        qwk_media=("qwk","mean"),qwk_sd=("qwk","std"),
        f1_macro_media=("f1_macro","mean"),
        recall_sii_3_medio=("recall_sii_3","mean"),n_pliegues=("qwk","size")
    ).reset_index()
    resumen_total.to_csv(RUTA_ABLACION/"02_resumen_cinco_bloques.csv",index=False)

    pares=[]
    for modelo in ("Logistica_multiclase","Boosting_histogramas"):
        ff=comparar[comparar.modelo==modelo]
        piv=ff.pivot_table(index=["repeticion","fold"],columns="bloque",values="qwk")
        for nom,prim,seg in [
            ("B - A","A_demografia","B_fisica"),
            ("C - B","B_fisica","C_ampliada"),
            ("B + SDS - B","B_fisica","B_mas_SDS"),
            ("B + Internet - B","B_fisica","B_mas_Internet")
        ]:
            for (rep,fold),delta in (piv[seg]-piv[prim]).items():
                pares.append({"modelo":modelo,"contraste":nom,"repeticion":rep,"fold":fold,"delta_qwk":float(delta)})
    pares = pd.DataFrame(pares)
    pares.to_csv(RUTA_ABLACION/"03_diferencias_pareadas_folds.csv",index=False)
    deltas = pares.groupby(["modelo","contraste"],sort=False).agg(
        diferencia_media=("delta_qwk","mean"),pliegues_positivos=("delta_qwk",lambda x:int((x>0).sum())),n=("delta_qwk","size")
    ).reset_index()
    deltas.to_csv(RUTA_ABLACION/"04_resumen_contrastes.csv",index=False)

    cm_ref = pd.read_csv(RUTA_MODELOS/"03_matrices_confusion_3_repeticiones.csv")
    precision_3=[]
    for bloque in ["A_demografia","B_fisica","C_ampliada",*bloques_ab]:
        for modelo in ("Logistica_multiclase","Boosting_histogramas"):
            if bloque in bloques_ab:
                cm = matrices_ab[(bloque,modelo)]
            else:
                v=cm_ref.query('cohorte=="5_17" and bloque==@bloque and modelo==@modelo')
                cm = np.zeros((4,4),dtype=int)
                for _,fila in v.iterrows():
                    cm[int(fila.real),int(fila.predicho)]=int(fila.conteo_en_3_repeticiones)
            precision_3.append({"bloque":bloque,"modelo":modelo,
                "precision_sii_3":float(cm[3,3]/cm[:,3].sum()) if cm[:,3].sum() else 0.0,
                "sensibilidad_sii_3":float(cm[3,3]/cm[3,:].sum()) if cm[3,:].sum() else 0.0,
                "verdaderos_positivos_3rep":int(cm[3,3]),
                "predichos_sii3_3rep":int(cm[:,3].sum()),"reales_sii3_3rep":int(cm[3,:].sum())})
    precision_3=pd.DataFrame(precision_3)
    precision_3.to_csv(RUTA_ABLACION/"05_precision_sii3_por_bloque.csv",index=False)

    # Referencia por clase mayoritaria (sii=0).
    pred_mayoria = np.zeros_like(y_ab)
    print("Modelo de mayoría sii=0 | QWK:",cohen_kappa_score(y_ab,pred_mayoria,weights="quadratic"),
          "| F1-macro:",f1_score(y_ab,pred_mayoria,average="macro",labels=[0,1,2,3],zero_division=0))
    print("\nComparación principal (cohorte 5–17):")
    print(resumen_total[["bloque","modelo","qwk_media","f1_macro_media","recall_sii_3_medio"]].round(4).to_string(index=False))
    print("\nContrastes pareados:")
    print(deltas.round(4).to_string(index=False))
    print("\nPrecisión/sensibilidad de clase severa:")
    print(precision_3.query('bloque=="C_ampliada"').round(4).to_string(index=False))
else:
    print("Ablaciones desactivadas en esta ejecución.")

## 9. Resultados de la comparación de predictores

| Bloque | QWK medio · Logística | QWK medio · Boosting |
|---|---:|---:|
| A | 0,325 | 0,284 |
| B | 0,323 | 0,291 |
| B + sueño | 0,368 | 0,362 |
| B + internet | 0,361 | 0,338 |
| C | 0,405 | 0,388 |

En la ejecución de referencia de la validación repetida de la cohorte de 5–17 años, el bloque C obtuvo el mayor kappa ponderado cuadrático (QWK) con ambos modelos. La precisión para identificar la categoría severa (`sii=3`) fue aproximadamente del 6,0 % con logística y del 5,4 % con *boosting*. Las matrices acumulan las predicciones de las tres repeticiones para los mismos participantes.


In [ ]:
if INCLUIR_ABLACIONES:
    for metrica,archivo in [("qwk_media","comparacion_qwk.png"),("f1_macro_media","comparacion_f1_macro.png")]:
        piv = resumen_total.pivot(index="bloque",columns="modelo",values=metrica)
        ax=piv.plot(kind="bar",figsize=(9,4.6),rot=20)
        ax.set_title("Validación cruzada interna — "+metrica)
        ax.set_ylabel(metrica)
        ax.set_xlabel("Bloque de predictores")
        ax.figure.tight_layout()
        ax.figure.savefig(RUTA_ABLACION/archivo,dpi=150)
        plt.show()
        plt.close(ax.figure)

    # Comparación del QWK del bloque C con la ejecución de referencia.
    esperado = {("C_ampliada","Logistica_multiclase"):0.404864,
                ("C_ampliada","Boosting_histogramas"):0.387721}
    for (b,m), valor in esperado.items():
        obtenido=float(resumen_total.query("bloque==@b and modelo==@m").qwk_media.iloc[0])
        print(f"{b} / {m}: obtenido {obtenido:.5f}; original {valor:.5f}; diferencia {obtenido-valor:+.5f}")
else:
    print(resumen_referencia[['cohorte','bloque','modelo','qwk_media','f1_macro_media']].round(4).to_string(index=False))

## 10. Exportación de resultados

Se exportaron tablas y gráficos agregados. El manifiesto registra la composición de la cohorte, los parámetros de evaluación, las versiones de las bibliotecas y la huella SHA-256 de `train.csv`.


In [ ]:
from datetime import datetime,timezone

registro = {
    "fecha_utc":datetime.now(timezone.utc).isoformat(),
    "proyecto":"MIAA 2026-2 PIU - EDA tabular y modelos",
    "competicion":COMPETICION,
    "fuente":"train.csv original de Kaggle/HBN",
    "sha256_train_csv":sha256_train,
    "filas_train":int(train.shape[0]),
    "columnas_train":int(train.shape[1]),
    "participantes_etiquetados":int(len(etiquetados)),
    "cohorte_5_17":int(len(principal)),
    "distribucion_sii_5_17":principal.sii.value_counts().sort_index().to_dict(),
    "cantidad_campos_PCIAT":len([c for c in train if c.startswith("PCIAT-")]),
    "predictores_bloques":BLOCKS,
    "semilla":2026,"validacion":"RepeatedStratifiedKFold 5 x 3",
    "cohortes_modeladas":COHORTES_EVALUAR,
    "ablaciones_exploratorias_ejecutadas":INCLUIR_ABLACIONES,
    "limitaciones":["Sin validación externa","Temporalidad tabular no comprobada",
      "Actigrafía excluida del entrenamiento", "Precisión muy baja en severidad alta",
      "Pliegues repetidos no independientes"],
    "versiones":{"python":platform.python_version(),"numpy":np.__version__,
                 "pandas":pd.__version__,"scipy":scipy.__version__,
                 "scikit_learn":sklearn.__version__}
}
(RUTA_RESULTADOS/"manifiesto.json").write_text(json.dumps(registro,ensure_ascii=False,indent=2),encoding="utf-8")

salida_zip=RUTA_BASE/"Resultados_PIU_MIAA_Notebook_Unificado.zip"
with zipfile.ZipFile(salida_zip,"w",compression=zipfile.ZIP_DEFLATED,compresslevel=7) as archivador:
    for archivo in sorted(RUTA_RESULTADOS.rglob("*")):
        if archivo.is_file():
            archivador.write(archivo,arcname=str(archivo.relative_to(RUTA_RESULTADOS)))
print("ZIP generado:",salida_zip,"| KB:",round(salida_zip.stat().st_size/1024,1))
print("ARCHIVOS DE SALIDA (agregados):")
for p in sorted(RUTA_RESULTADOS.rglob("*")):
    if p.is_file(): print(" -",p.relative_to(RUTA_RESULTADOS))

if DESCARGAR_ZIP_AL_FINAL and EN_COLAB:
    from google.colab import files
    files.download(str(salida_zip))

### Alcance del análisis

El análisis retrospectivo se limitó a variables tabulares y dejó fuera las series de actigrafía. Las categorías de `sii` corresponden a la escala del conjunto Healthy Brain Network. La disponibilidad de los predictores antes del desenlace y el desempeño en una cohorte independiente quedan por evaluar.


## 11. Evaluación ampliada de los modelos

Con el bloque C en la cohorte de 5–17 años se compararon siete estrategias. Se incluyeron la regresión logística y el *boosting* de referencia, sus versiones optimizadas, sus versiones con calibración sigmoide y un clasificador ordinal acumulativo.

El QWK fue la métrica principal. Se complementó con F1-macro, exactitud balanceada, error absoluto medio ordinal, proporción de errores de dos o más categorías, precisión y sensibilidad para `sii=3`, Brier multiclase y *log-loss*.


In [ ]:
# Parámetros de la validación anidada.
EJECUTAR_EVALUACION_AMPLIADA = True
SEMILLA_AVANZADA = 2026
PLIEGUES_EXTERNOS = 5
PLIEGUES_INTERNOS = 3
MUESTRAS_BOOTSTRAP = 400  # Remuestreo de predicciones fuera de muestra.
EMPLEOS_PARA_BUSQUEDA = 2  # Procesos paralelos para la búsqueda.
DESCARGAR_RESULTADOS_AVANZADOS = True  # Descarga el ZIP consolidado al terminar.

RUTA_AVANZADA = RUTA_RESULTADOS / "evaluacion_avanzada"
RUTA_AVANZADA.mkdir(parents=True, exist_ok=True)

if EJECUTAR_EVALUACION_AMPLIADA:
    assert set(["Basic_Demos-Age", "Basic_Demos-Sex", "SDS-SDS_Total_T", "PreInt_EduHx-computerinternet_hoursday"]).issubset(train.columns)
    assert PLIEGUES_EXTERNOS >= 2 and PLIEGUES_INTERNOS >= 2
    assert not any(x.startswith("PCIAT-") or x in ("id", "sii") for x in BLOCKS["C_ampliada"])
    print("Validación anidada · pliegues externos:",PLIEGUES_EXTERNOS,"| pliegues internos:",PLIEGUES_INTERNOS,"| conjunto C, edad 5–17 años")


### 11.1. Clasificador ordinal acumulativo

Se ajustaron tres regresiones logísticas para estimar la probabilidad de superar cada umbral de severidad (`sii > 0`, `sii > 1` y `sii > 2`). Se aplicó una corrección de monotonía a esas probabilidades para obtener la probabilidad de cada una de las cuatro categorías.


In [ ]:
from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.metrics import log_loss, mean_absolute_error
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.metrics import make_scorer
from sklearn.calibration import CalibratedClassifierCV
from sklearn.utils.validation import check_is_fitted

class RegresionOrdinalAcumulativa(ClassifierMixin, BaseEstimator):
    """Tres regresiones logísticas para P(y > k), con corrección de monotonía."""
    def __init__(self, C=1.0, class_weight=None, random_state=2026):
        self.C=C
        self.class_weight=class_weight
        self.random_state=random_state

    def fit(self, X, y):
        y=np.asarray(y, dtype=int)
        if set(np.unique(y)) != {0,1,2,3}:
            raise ValueError("El entrenamiento ordinal requiere las cuatro categorías sii 0,1,2,3.")
        self.classes_=np.array([0,1,2,3], dtype=int)
        self.modelos_=[]
        for umbral in (0,1,2):
            modelo=LogisticRegression(C=self.C,
                 class_weight=self.class_weight,max_iter=1800,random_state=self.random_state)
            modelo.fit(X,(y>umbral).astype(int))
            self.modelos_.append(modelo)
        return self

    def predict_proba(self, X):
        check_is_fitted(self, "modelos_")
        acumuladas=np.column_stack([m.predict_proba(X)[:,1] for m in self.modelos_])
        acumuladas=np.minimum.accumulate(acumuladas,axis=1)
        p0=1.0-acumuladas[:,0]
        p1=acumuladas[:,0]-acumuladas[:,1]
        p2=acumuladas[:,1]-acumuladas[:,2]
        p3=acumuladas[:,2]
        probs=np.clip(np.column_stack([p0,p1,p2,p3]),0.0,1.0)
        return probs/np.maximum(probs.sum(axis=1,keepdims=True),1e-12)

    def predict(self, X):
        return self.classes_[np.argmax(self.predict_proba(X),axis=1)]

def crear_pipeline_ordinal():
    base=mk_pipeline(BLOCKS["C_ampliada"], "Logistica_multiclase")
    return Pipeline([("preprocess",clone(base.named_steps["preprocess"])),
                     ("model",RegresionOrdinalAcumulativa())])

def metricas_completas(y, pred, proba):
    y=np.asarray(y,dtype=int); pred=np.asarray(pred,dtype=int)
    proba=np.asarray(proba,dtype=float)
    assert proba.shape == (len(y),4) and np.isfinite(proba).all()
    assert np.allclose(proba.sum(axis=1),1.0,atol=1e-6)
    cm=confusion_matrix(y,pred,labels=[0,1,2,3])
    clase3_real=int(cm[3,:].sum()); clase3_pred=int(cm[:,3].sum())
    onehot=np.eye(4)[y]
    return {
        "qwk":float(cohen_kappa_score(y,pred,weights="quadratic",labels=[0,1,2,3])),
        "f1_macro":float(f1_score(y,pred,labels=[0,1,2,3],average="macro",zero_division=0)),
        "exactitud_balanceada":float(balanced_accuracy_score(y,pred)),
        "mae_ordinal":float(mean_absolute_error(y,pred)),
        "errores_ordinales_graves_pct":float(100*np.mean(np.abs(y-pred)>=2)),
        "precision_sii_3":float(cm[3,3]/clase3_pred) if clase3_pred else 0.0,
        "sensibilidad_sii_3":float(cm[3,3]/clase3_real) if clase3_real else 0.0,
        "predicciones_sii_3":clase3_pred,
        "verdaderos_positivos_sii_3":int(cm[3,3]),
        "log_loss":float(log_loss(y,np.clip(proba,1e-12,1.0),labels=[0,1,2,3])),
        "brier_multiclase":float(np.mean(np.sum((onehot-proba)**2,axis=1))),
    }

ESPACIO_LOGISTICA={
    "model__C":[0.1,1.0,5.0],
    "model__class_weight":[None,"balanced"],
}
ESPACIO_BOOSTING={
    "model__learning_rate":[0.04,0.07],
    "model__max_leaf_nodes":[7,15],
    "model__class_weight":[None,"balanced"],
}
PUNTUACION_QWK=make_scorer(cohen_kappa_score,weights="quadratic",labels=[0,1,2,3])
print("Combinaciones de hiperparámetros:",
      6,"logística;",8,"boosting")


### 11.2. Validación cruzada anidada

Las siete estrategias se evaluaron en los mismos cinco pliegues externos estratificados. Dentro del conjunto de entrenamiento de cada pliegue externo se usaron tres pliegues internos para seleccionar hiperparámetros por QWK o ajustar la calibración. Cada participante aportó una predicción fuera de muestra (`OOF`) por estrategia. Las comparaciones pareadas de QWK se calcularon sobre esas predicciones, con percentiles de remuestreo condicionados a las predicciones de esta validación.


In [ ]:
def evaluar_evaluacion_ampliada():
    if not EJECUTAR_EVALUACION_AMPLIADA:
        print("Evaluación ampliada omitida por configuración.")
        return None

    data=basic_quality_clean(principal)
    X=data[BLOCKS["C_ampliada"]].reset_index(drop=True)
    y=data["sii"].astype(int).to_numpy()
    n=len(y)
    assert n==2652 and np.bincount(y,minlength=4).tolist()==[1562,700,360,30]
    externa=StratifiedKFold(n_splits=PLIEGUES_EXTERNOS,shuffle=True,random_state=SEMILLA_AVANZADA)
    interna=StratifiedKFold(n_splits=PLIEGUES_INTERNOS,shuffle=True,random_state=SEMILLA_AVANZADA)

    nombres=["Logistica_original","Logistica_optimizada",
             "Boosting_original","Boosting_optimizado",
             "Ordinal_acumulativa","Logistica_calibrada","Boosting_calibrado"]
    preds_oof={nombre:np.full(n,-1,dtype=int) for nombre in nombres}
    probas_oof={nombre:np.full((n,4),np.nan) for nombre in nombres}
    registros=[]; ajustes=[]

    for num_fold,(it,iv) in enumerate(externa.split(X,y),1):
        x_ent,x_val=X.iloc[it],X.iloc[iv]
        y_ent,y_val=y[it],y[iv]
        print(f"--- Pliegue externo {num_fold}/{PLIEGUES_EXTERNOS}: "
              f"entrenamiento {len(it)}, validación {len(iv)}, casos severos de validación {(y_val==3).sum()} ---")
        estimadores={}
        for etiqueta,algoritmo,rejilla in [
            ("Logistica","Logistica_multiclase",ESPACIO_LOGISTICA),
            ("Boosting","Boosting_histogramas",ESPACIO_BOOSTING)
        ]:
            base=mk_pipeline(BLOCKS["C_ampliada"],algoritmo)
            base.fit(x_ent,y_ent)
            estimadores[f"{etiqueta}_original"]=base

            buscador=GridSearchCV(
                estimator=mk_pipeline(BLOCKS["C_ampliada"],algoritmo),
                param_grid=rejilla,scoring=PUNTUACION_QWK,
                cv=interna,n_jobs=EMPLEOS_PARA_BUSQUEDA,refit=True,
                error_score="raise",return_train_score=False)
            buscador.fit(x_ent,y_ent)
            estimadores[f"{etiqueta}_{'optimizada' if etiqueta=='Logistica' else 'optimizado'}"]=buscador.best_estimator_
            ajustes.append({"pliegue_externo":num_fold,"modelo":etiqueta,
                            "mejor_qwk_interno":float(buscador.best_score_),
                            "hiperparametros":json.dumps(buscador.best_params_,sort_keys=True)})

            # El calibrador se ajusta únicamente con el entrenamiento del pliegue externo.
            calibrador=CalibratedClassifierCV(
                estimator=mk_pipeline(BLOCKS["C_ampliada"],algoritmo),
                method="sigmoid",cv=interna,ensemble=False)
            calibrador.fit(x_ent,y_ent)
            estimadores[f"{etiqueta}_{'calibrada' if etiqueta=='Logistica' else 'calibrado'}"]=calibrador

        ordinal=crear_pipeline_ordinal()
        ordinal.fit(x_ent,y_ent)
        estimadores["Ordinal_acumulativa"]=ordinal

        for nombre,modelo in estimadores.items():
            probas=modelo.predict_proba(x_val)
            pred=modelo.predict(x_val)
            # Orden 0..3 presente en todos los entrenamientos externos.
            clases=getattr(modelo,"classes_",None)
            if clases is None and hasattr(modelo,"named_steps"):
                clases=getattr(modelo.named_steps["model"],"classes_",None)
            assert np.array_equal(np.asarray(clases),np.array([0,1,2,3])),(nombre,clases)
            resultado=metricas_completas(y_val,pred,probas)
            registros.append({"modelo":nombre,"pliegue_externo":num_fold,
                              "n_entrenamiento":len(it),"n_validacion":len(iv),
                              "severos_validacion":int((y_val==3).sum()),**resultado})
            assert np.all(preds_oof[nombre][iv]==-1),"Predicción externa duplicada"
            preds_oof[nombre][iv]=pred
            probas_oof[nombre][iv]=probas
        print("Comparaciones externas completadas para este pliegue.")

    por_fold=pd.DataFrame(registros)
    assert len(por_fold)==len(nombres)*PLIEGUES_EXTERNOS
    por_fold.to_csv(RUTA_AVANZADA/"01_metricas_por_fold_externo.csv",index=False)
    pd.DataFrame(ajustes).to_csv(RUTA_AVANZADA/"02_hiperparametros_por_fold.csv",index=False)

    resumen=[]; matrices=[]
    for nombre in nombres:
        assert (preds_oof[nombre]>=0).all() and np.isfinite(probas_oof[nombre]).all()
        m=metricas_completas(y,preds_oof[nombre],probas_oof[nombre])
        dfm=por_fold.loc[por_fold.modelo==nombre]
        resumen.append({"modelo":nombre,"n_participantes":n,
                        "n_folds_externos":PLIEGUES_EXTERNOS,
                        **{f"oof_{k}":v for k,v in m.items()},
                        "qwk_media_folds":float(dfm.qwk.mean()),
                        "qwk_sd_folds_descriptiva":float(dfm.qwk.std())})
        cm=confusion_matrix(y,preds_oof[nombre],labels=[0,1,2,3])
        for real in range(4):
            for predicho in range(4):
                matrices.append({"modelo":nombre,"real":real,"predicho":predicho,
                                 "conteo":int(cm[real,predicho])})
    resumen_df=pd.DataFrame(resumen).sort_values("oof_qwk",ascending=False).reset_index(drop=True)
    resumen_df.to_csv(RUTA_AVANZADA/"03_resumen_oof_5_folds.csv",index=False)
    pd.DataFrame(matrices).to_csv(RUTA_AVANZADA/"04_matrices_confusion_agregadas.csv",index=False)

    # Diferencias de QWK sobre las mismas predicciones fuera de muestra por participante.
    comparaciones=[("Logistica_optimizada","Logistica_original"),
                   ("Boosting_optimizado","Boosting_original"),
                   ("Ordinal_acumulativa","Logistica_original"),
                   ("Logistica_calibrada","Logistica_original"),
                   ("Boosting_calibrado","Boosting_original")]
    rng=np.random.default_rng(SEMILLA_AVANZADA)
    por_clase=[np.flatnonzero(y==k) for k in range(4)]
    contrastes=[]
    for mejor,referencia in comparaciones:
        delta_observado=(cohen_kappa_score(y,preds_oof[mejor],labels=[0,1,2,3],weights="quadratic")-
                         cohen_kappa_score(y,preds_oof[referencia],labels=[0,1,2,3],weights="quadratic"))
        boot=[]
        for _ in range(MUESTRAS_BOOTSTRAP):
            idx=np.concatenate([rng.choice(g,size=len(g),replace=True) for g in por_clase])
            rng.shuffle(idx)
            a=cohen_kappa_score(y[idx],preds_oof[mejor][idx],labels=[0,1,2,3],weights="quadratic")
            b=cohen_kappa_score(y[idx],preds_oof[referencia][idx],labels=[0,1,2,3],weights="quadratic")
            boot.append(float(a-b))
        lo,hi=np.quantile(boot,[.025,.975])
        contrastes.append({"estrategia":mejor,"comparador":referencia,
                           "delta_qwk_oof":float(delta_observado),
                           "percentil_boot_2_5_condicional":float(lo),
                           "percentil_boot_97_5_condicional":float(hi),
                           "n_bootstrap":MUESTRAS_BOOTSTRAP})
    pd.DataFrame(contrastes).to_csv(RUTA_AVANZADA/"05_contrastes_pareados_bootstrap_descriptivo.csv",index=False)

    meta={"titulo":"Evaluación ampliada exploratoria de modelos HBN, cohorte 5–17",
          "sha256_train_csv":sha256_train,"n_cohorte":n,
          "cv_externa":{"tipo":"StratifiedKFold","n_splits":PLIEGUES_EXTERNOS,"shuffle":True,"seed":SEMILLA_AVANZADA},
          "cv_interna":{"tipo":"StratifiedKFold","n_splits":PLIEGUES_INTERNOS,"shuffle":True,"seed":SEMILLA_AVANZADA},
          "optimizacion":{"metrica":"QWK cuadratico","regresion_logistica":ESPACIO_LOGISTICA,
                          "boosting":ESPACIO_BOOSTING},
          "sistemas_evaluados":nombres,
          "bootstrap":{"n":MUESTRAS_BOOTSTRAP,"estratificado_por_clase":True,
             "limitacion":"Percentiles descriptivos condicionados a las predicciones fuera de muestra."},
          "notas":["Preprocesamiento siempre incluido en Pipeline de entrenamiento",
                   "Optimizacion anidada: nunca examina etiquetas del pliegue externo",
                   "La calibracion se ajusta solo con entrenamiento externo",
                   "Los CSV contienen solo métricas agregadas, sin IDs ni predicciones por persona",
                   "La clase sii=3 tiene solo 30 participantes; alto riesgo de inestabilidad",
                   "Validación interna retrospectiva",
                   "Las métricas OOF corresponden a la validación anidada"],
          "versiones":{"python":platform.python_version(),"pandas":pd.__version__,
                       "numpy":np.__version__,"sklearn":sklearn.__version__}}
    (RUTA_AVANZADA/"06_protocolo_y_versiones.json").write_text(
        json.dumps(meta,ensure_ascii=False,indent=2),encoding="utf-8")
    print("\nResultados fuera de muestra · cohorte de 5–17 años:")
    print(resumen_df[["modelo","oof_qwk","oof_f1_macro","oof_mae_ordinal",
                      "oof_precision_sii_3","oof_sensibilidad_sii_3","oof_brier_multiclase"]].round(4).to_string(index=False))
    print("\nComparaciones pareadas de QWK y percentiles de remuestreo:")
    print(pd.DataFrame(contrastes).round(4).to_string(index=False))
    return {"resumen":resumen_df,"por_fold":por_fold,
            "oof_pred":preds_oof,"oof_proba":probas_oof,
            "y":y,"contrastes":pd.DataFrame(contrastes)}

resultado_avanzado=evaluar_evaluacion_ampliada()


### 11.3. Desempeño ordinal y detección de casos severos

Se examinó el desempeño global junto con la capacidad de detectar `sii=3`. El puntaje de Brier y el *log-loss* permitieron evaluar el efecto de la calibración sobre la calidad de las probabilidades.


In [ ]:
if resultado_avanzado is not None:
    resumen_v2=resultado_avanzado["resumen"]
    eje=resumen_v2.sort_values("oof_qwk")
    fig,ax=plt.subplots(figsize=(10,5))
    ax.barh(eje.modelo,eje.oof_qwk)
    ax.set_xlabel("QWK en predicciones fuera de muestra (validación anidada)")
    ax.set_title("Comparación exploratoria de estrategias — bloque C")
    fig.tight_layout();fig.savefig(RUTA_AVANZADA/"07_comparacion_qwk.png",dpi=170);plt.show();plt.close(fig)

    fig,ax=plt.subplots(figsize=(10,5))
    ax.scatter(resumen_v2.oof_precision_sii_3,resumen_v2.oof_sensibilidad_sii_3,s=65)
    for _,fila in resumen_v2.iterrows():
        ax.annotate(fila.modelo,(fila.oof_precision_sii_3,fila.oof_sensibilidad_sii_3),
                    fontsize=8,xytext=(4,4),textcoords="offset points")
    ax.set_xlabel("Precisión sii=3")
    ax.set_ylabel("Sensibilidad sii=3")
    ax.set_title("Compromiso entre falsas alertas y casos severos detectados")
    fig.tight_layout();fig.savefig(RUTA_AVANZADA/"08_tradeoff_severidad.png",dpi=170);plt.show();plt.close(fig)

    print("Brier multiclase y log-loss por estrategia:")
    print(resumen_v2[["modelo","oof_brier_multiclase","oof_log_loss"]].sort_values("oof_brier_multiclase").round(4).to_string(index=False))
else:
    print("Evaluación ampliada sin ejecutar.")


### 11.4. Resultados exportados

Cada matriz de confusión consolida 2.652 predicciones. El archivo consolidado incluye métricas por pliegue, hiperparámetros seleccionados, matrices agregadas, comparaciones pareadas y gráficos.


In [ ]:
if resultado_avanzado is not None:
    matrices_v2=pd.read_csv(RUTA_AVANZADA/"04_matrices_confusion_agregadas.csv")
    assert (matrices_v2.groupby("modelo").conteo.sum()==2652).all(),         "Cada matriz debe sumar una predicción fuera de muestra por participante."
    archivos_publicables=sorted(str(p.relative_to(RUTA_RESULTADOS))
                                for p in RUTA_AVANZADA.iterdir() if p.is_file())
    assert not any("train.csv" in p or "kaggle.json" in p for p in archivos_publicables)
    zip_final=RUTA_BASE/"Resultados_PIU_MIAA_Con_Evaluacion_Ampliada.zip"
    with zipfile.ZipFile(zip_final,"w",compression=zipfile.ZIP_DEFLATED,compresslevel=7) as z:
        for archivo in sorted(RUTA_RESULTADOS.rglob("*")):
            if archivo.is_file():
                z.write(archivo,arcname=str(archivo.relative_to(RUTA_RESULTADOS)))
    print("Evaluación avanzada exportada:",zip_final.name)
    print("Archivos de sección avanzada:")
    for ar in archivos_publicables: print(" -",ar)
    if DESCARGAR_RESULTADOS_AVANZADOS and EN_COLAB:
        from google.colab import files
        files.download(str(zip_final))
else:
    print("Exportación de la evaluación ampliada sin resultados.")


## 12. Resultados de la evaluación ampliada

La ejecución del 14 de noviembre de 2026 utilizó el mismo `train.csv` que el análisis de referencia, según su huella SHA-256. La tabla resume el desempeño sobre las predicciones fuera de muestra del protocolo anidado. Los QWK de la sección 9 son medias de pliegues de la validación repetida.


| Estrategia | QWK | F1-macro | Precisión `sii=3` | Sensibilidad `sii=3` | Brier ↓ |
|---|---:|---:|---:|---:|---:|
| Boosting de referencia | 0,401 | 0,372 | 5,9 % | 10,0 % | 0,569 |
| Boosting optimizado | 0,425 | 0,397 | 10,7 % | 40,0 % | 0,573 |
| Logística de referencia | 0,400 | 0,357 | 5,6 % | 60,0 % | 0,599 |
| Logística optimizada | 0,400 | 0,357 | 5,7 % | 60,0 % | 0,599 |
| Ordinal acumulativo | 0,355 | 0,350 | 100,0 % | 3,3 % | 0,492 |
| Logística calibrada | 0,197 | 0,259 | 100,0 % | 3,3 % | 0,516 |
| Boosting calibrado | 0,153 | 0,268 | 50,0 % | 3,3 % | 0,514 |

El *boosting* optimizado obtuvo el mayor QWK y mejoró 0,024 puntos frente a su versión de referencia. Detectó 12 de los 30 casos severos, frente a 3 con el modelo de referencia. Las falsas alertas aumentaron de 48 a 100. El ajuste de hiperparámetros de la regresión logística produjo un cambio mínimo en QWK.

La calibración mejoró Brier y *log-loss* en ambos modelos y redujo el QWK al asignar la clase de mayor probabilidad. El modelo ordinal obtuvo el menor error absoluto medio ordinal y el menor Brier, con un solo caso severo detectado. El bajo número de casos con `sii=3` limita la estabilidad de las estimaciones de precisión y sensibilidad.
